# 17 · MrVI pseudo-sample malignancy classifier

TCR-anchored **malignant vs benign** T cells via a **pseudo-sample MrVI** (`data/build_method3_pseudosample_mrvi.md`), occupying the **Method 2** slot (`prob_m2 / call_m2 / uncertain_m2`). Each donor is split into `{donor}_{malignant|benign|unknown}` pseudo-samples; a fresh MrVI is trained with `pseudo_sample` as the sample covariate (batch = `study`) on **all** T cells. Cells are then scored per Leiden cluster by distance to that cluster's malignant vs benign anchor centroid in MrVI `z` space — donor-aware and batch-corrected, a cleaner estimate of the same axis as the old Option 2A.

- **Anchors** reuse the canonical per-donor dominant TCR clone (mirrors nb14/15), identical to nb16.
- Heavy train runs as a bsub job (`jobs/run_mrvi_pseudosample_m2.sh`); self-contained; shares `data/atlas_joint/mrvi_malignancy/calls.csv` with `16_mrvi_labelspreading_malignancy`.
- No-TCR cells are **scored** (never dropped); HC donors are clamped benign; `uncertain_m2` flags the early-stage overlap band (`prob ∈ [0.4,0.6]` or `|s| < MARGIN_MIN`).

In [ ]:
# ============================ CONFIG (shared with nb16 except method params) ============================
import sys, gc, warnings
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(start)


NB_DIR = _resolve_nb_dir()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
import importlib
import atlas_join_helpers as H
import skin_T_cnv_helpers as C
importlib.reload(H); importlib.reload(C)

SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1

# ---- paths ----
V2        = NB_DIR / "data/atlas_joint/skin_T_tcr_malig_v2.h5ad"       # curated TCR-complete T object (has X_mrvi_u)
HVG_INPUT = NB_DIR / "data/atlas_joint/joint_mrvi_input.h5ad"          # MrVI HVG var set
UMAP_NPY  = NB_DIR / "data/atlas_joint/skin_T_umap_u.npy"              # precomputed u-UMAP
OUT       = NB_DIR / "data/atlas_joint/mrvi_malignancy"; OUT.mkdir(parents=True, exist_ok=True)
CALLS_CSV = OUT / "calls.csv"

# ---- pseudo-sample MrVI job artifacts (written by jobs/run_mrvi_pseudosample_m2.py) ----
M2_INPUT        = OUT / "m2_pseudosample_input.h5ad"
M2_MODEL_DIR    = NB_DIR / "models/mrvi_pseudosample_m2"
M2_U_NPY        = OUT / "m2_u.npy"
M2_Z_NPY        = OUT / "m2_z.npy"
M2_BC_NPY       = OUT / "m2_barcodes.npy"
M2_CLUSTERS_NPY = OUT / "m2_clusters.npy"
M2_JOB_SH       = NB_DIR / "jobs/run_mrvi_pseudosample_m2.sh"
M2_JOB_LOG      = NB_DIR / "jobs/run_mrvi_pseudosample_m2.bsub.log"

# ---- obs / obsm keys ----
U_KEY = "X_mrvi_u"; SAMPLE_KEY = "donor"; STUDY_KEY = "study"; STAGE_KEY = "stage_class"; LINEAGE_KEY = "lineage"

# ---- anchors: malignant = carried ALICE call; benign = TCR+ non-malignant unexpanded ----
MIN_ANCHORS = 30

# ---- method params (pseudo-sample MrVI + per-cluster centroid scoring) ----
MIN_PSEUDO = 20        # min cells for a malignant/benign pseudo-sample (else folded into _unknown)
MARGIN_MIN = 0.02      # abstain band on |score|
USE_REP    = "z"       # centroid/scoring space: "z" (sample-aware) or "u" (sample-unaware fallback)

# ---- M2 post-job cache (anchor CV + final calls; MrVI train/Leiden cached separately on disk) ----
REFIT_M2 = True                        # True -> recompute (required: v2 cell set + anchors changed)
M2_CACHE = OUT / "nb17_m2_cache.npz"   # {thr_m2, prob_m2, call_m2, uncertain_m2} keyed by barcode

In [ ]:
# ============================ SHARED PREPROCESSING (identical in nb16/nb17) ============================
# ---- load the curated TCR-complete T object (242,959 cells; carries ALICE call + X_mrvi_u) ----
adata = sc.read_h5ad(V2)
# ALICE tumor call is the malignant truth; it also drives the benign anchor exclusion below
alice = adata.obs["tcr_malignant_alice"].astype(bool).to_numpy()
adata.obs["tcr_is_malignant"]      = alice
adata.obs["tcr_is_dominant_clone"] = alice
# tcr_is_expanded / has_tcr / tcr_clone_size / tcr_clone_id come from the file

# ---- decision gate: required obs columns + u embedding ----
need = [SAMPLE_KEY, STUDY_KEY, LINEAGE_KEY, "has_tcr", "tra_cdr3", "trb_cdr3"]
miss = [c for c in need if c not in adata.obs.columns]
stage_key = (STAGE_KEY if STAGE_KEY in adata.obs.columns
             else "disease_stage" if "disease_stage" in adata.obs.columns else None)
if miss or U_KEY not in adata.obsm or stage_key is None:
    print("obs.columns:", list(adata.obs.columns))
    print("obsm.keys :", list(adata.obsm.keys()))
    raise SystemExit(f"DECISION GATE — missing cols={miss}  u_present={U_KEY in adata.obsm}  stage_key={stage_key}")
if "cell_type_broad" in adata.obs.columns:
    assert adata.obs["cell_type_broad"].astype(str).eq("T").all(), "object should be all T cells"
print(f"stage_key = {stage_key}  |  cells = {adata.n_obs}  |  u dim = {adata.obsm[U_KEY].shape[1]}")

# ---- anchors: malignant = ALICE call; benign = TCR+ non-malignant, unexpanded clone ----
o = adata.obs
has = o["has_tcr"].astype(bool).to_numpy()
mal = o["tcr_is_malignant"].astype(bool).to_numpy()
ben = has & ~mal & ~o["tcr_is_expanded"].astype(bool).to_numpy()
lab = np.where(mal, "malignant", np.where(ben, "benign", "unlabeled"))   # no-TCR -> unlabeled, never benign
adata.obs["tcr_label"] = pd.Categorical(lab, categories=["malignant", "benign", "unlabeled"])

vc = adata.obs["tcr_label"].value_counts()
print("\nanchor counts:\n", vc.to_string())
for cls in ("malignant", "benign"):
    if int(vc.get(cls, 0)) < MIN_ANCHORS:
        warnings.warn(f"class '{cls}' has {int(vc.get(cls, 0))} < MIN_ANCHORS={MIN_ANCHORS}")
print("\nanchors per study:\n",
      adata.obs.groupby(STUDY_KEY, observed=True)["tcr_label"]
      .value_counts().unstack(fill_value=0).to_string())

# ---- u embedding + UMAP on u (recompute if the cached npy is stale for this cell set) ----
U = np.asarray(adata.obsm[U_KEY])
if UMAP_NPY.exists() and np.load(UMAP_NPY).shape[0] == adata.n_obs:
    adata.obsm["X_umap_u"] = np.load(UMAP_NPY)
    print("\nloaded u-UMAP:", UMAP_NPY.name)
else:
    sc.pp.neighbors(adata, use_rep=U_KEY, random_state=SEED)   # HEAVY
    sc.tl.umap(adata, random_state=SEED)
    adata.obsm["X_umap_u"] = adata.obsm["X_umap"]
    np.save(UMAP_NPY, adata.obsm["X_umap_u"]); print("\ncomputed + saved u-UMAP ->", UMAP_NPY.name)

# ---- handoff base into shared calls.csv (index = barcode) ----
base = pd.DataFrame({
    SAMPLE_KEY: o[SAMPLE_KEY].astype(str).values,
    STUDY_KEY:  o[STUDY_KEY].astype(str).values,
    "stage":    o[stage_key].astype(str).values,
    LINEAGE_KEY: o[LINEAGE_KEY].astype(str).values,
    "tcr_label": adata.obs["tcr_label"].astype(str).values,
    "clone_size": o["tcr_clone_size"].astype(int).values,
}, index=adata.obs_names)
base.index.name = "barcode"
if CALLS_CSV.exists():
    prev = pd.read_csv(CALLS_CSV, index_col=0)
    for c in base.columns:
        prev.loc[base.index, c] = base[c]      # refresh base cols, keep any prob_*/call_* from the other nb
    base = prev
base.to_csv(CALLS_CSV)
print("calls base ->", CALLS_CSV, base.shape)

## Pseudo-samples + train pseudo-sample MrVI

Split each donor into TCR-anchored pseudo-samples (`{donor}_{malignant|benign|unknown}`), apply the `MIN_PSEUDO` size gate (small malignant/benign pseudo-samples fold into `_unknown`), and write a slim job input (raw counts on the MrVI HVG set + `pseudo_sample`/`malignant_status`). The heavy MrVI train (`sample_key=pseudo_sample`, `batch_key=study`, all T cells incl. `_unknown`) runs as a bsub job, which persists per-cell `u` and `z`.

In [ ]:
# ---- build TCR-anchored pseudo-samples + write the slim bsub job input ----
import anndata as ad

status = adata.obs["tcr_label"].astype(str).map(
    {"malignant": "malignant", "benign": "benign"}).fillna("unknown").to_numpy()
donor = adata.obs[SAMPLE_KEY].astype(str).to_numpy()         # object dtype -> elementwise str +
pseudo = donor + "_" + status

# size gate: malignant/benign pseudo-samples below MIN_PSEUDO -> fold into {donor}_unknown
ps_counts = pd.Series(pseudo).value_counts()
small = {p for p in ps_counts.index if ps_counts[p] < MIN_PSEUDO and not p.endswith("_unknown")}
demote = np.isin(pseudo, list(small))
status[demote] = "unknown"
pseudo[demote] = donor[demote] + "_unknown"
print(f"demoted {int(demote.sum())} cells from {len(small)} sub-{MIN_PSEUDO} pseudo-samples -> _unknown")

adata.obs["pseudo_sample"] = pd.Categorical(pseudo)
adata.obs["malignant_status"] = pd.Categorical(status, categories=["benign", "malignant", "unknown"])

# per-donor contribution to the contrast axis
ct = adata.obs.groupby([SAMPLE_KEY, "malignant_status"], observed=True).size().unstack(fill_value=0)
def _side(r):
    m, b = r.get("malignant", 0) > 0, r.get("benign", 0) > 0
    return "both" if m and b else "malignant-only" if m else "benign-only" if b else "none"
print("\nper-donor contrast contribution:\n", ct.apply(_side, axis=1).value_counts().to_string())
print("\nmalignant_status:\n", adata.obs["malignant_status"].value_counts().to_string())
no_tcr = ~adata.obs["has_tcr"].astype(bool).to_numpy()
print(f"no-TCR cells routed to _unknown: {int((status[no_tcr] == 'unknown').sum())}/{int(no_tcr.sum())}")

# ---- slim job input: raw_counts on the MrVI HVG set + the covariates MrVI needs ----
if not HVG_INPUT.exists():
    raise SystemExit(f"missing MrVI HVG set {HVG_INPUT}")
hvg = set(sc.read_h5ad(HVG_INPUT, backed="r").var_names)
gmask = adata.var_names.isin(hvg)
sub = adata[:, gmask]
inp = ad.AnnData(
    X=sub.layers["raw_counts"].copy(),
    obs=adata.obs[[SAMPLE_KEY, STUDY_KEY, "pseudo_sample", "malignant_status", "tcr_label"]].copy(),
    var=pd.DataFrame(index=sub.var_names.copy()),
)
inp.layers["raw_counts"] = inp.X.copy()
inp.write_h5ad(M2_INPUT)
print(f"\nwrote job input -> {M2_INPUT}  {inp.shape}  ({int(gmask.sum())} HVGs)")
del inp, sub; gc.collect()

In [ ]:
# ---- submit the pseudo-sample MrVI bsub job (skip if latents already on disk) ----
import subprocess
if M2_Z_NPY.exists() and M2_U_NPY.exists() and M2_BC_NPY.exists():
    print("latents already present; skipping submit ->", M2_Z_NPY.name)
else:
    r = subprocess.run(["bash", str(M2_JOB_SH)], capture_output=True, text=True)
    print(r.stdout, end=""); print(r.stderr, end="")
    print("\nlog:", M2_JOB_LOG)
    print("poll the next cell (re-run) until the artifacts appear, then continue")

In [ ]:
# ---- poll: re-run until the job artifacts exist (m2_z.npy / m2_u.npy / m2_barcodes.npy) ----
import subprocess
print("=== bjobs -a (mrvi_psm2) ===")
print(subprocess.run(["bjobs", "-a", "-J", "mrvi_psm2"], capture_output=True, text=True).stdout or "(no jobs)")
for p in (M2_MODEL_DIR / "model.pt", M2_Z_NPY, M2_U_NPY, M2_BC_NPY):
    print(f"  {'OK ' if Path(p).exists() else '...'} {p.name}")
if M2_JOB_LOG.exists():
    print("\n=== tail", M2_JOB_LOG.name, "===")
    print(subprocess.run(["tail", "-n", "25", str(M2_JOB_LOG)], capture_output=True, text=True).stdout)

## Cluster + score (per-cluster anchor centroids)

Load the per-cell `u`/`z` latents from the job. Leiden-cluster on `u` (sample-unaware → malignancy-agnostic). Per cluster `c`: `mal_z[c]` / `ben_z[c]` = mean `R` over malignant / benign anchors in `c` (fallback to the global class centroid where a cluster lacks one class). Every cell — anchor, unknown, or no-TCR — is scored `s = ‖R − ben_z[c]‖ − ‖R − mal_z[c]‖` (>0 → malignant), then a 1-D logistic on anchors maps `s → prob_m2`. `R = z` (or `u` if `USE_REP="u"`).

In [ ]:
# ---- load pseudo-sample MrVI latents (from the bsub job), reindexed to adata ----
from sklearn.linear_model import LogisticRegression

bc = np.load(M2_BC_NPY, allow_pickle=True).astype(str)
assert len(bc) == adata.n_obs, (   # guard against stale latents from a previous (e.g. 401k) cell set
    f"m2 latents have {len(bc)} cells != adata {adata.n_obs} — delete m2_*.npy and re-run the bsub job")
order = pd.Index(bc).get_indexer(adata.obs_names)
assert (order >= 0).all(), "barcode mismatch between job output and adata"
u = np.load(M2_U_NPY)[order]
z = np.load(M2_Z_NPY)[order]
R = z if USE_REP == "z" else u
print(f"latents: z {z.shape}  u {u.shape}  | scoring on USE_REP={USE_REP!r}")

# ---- Leiden clusters on the pseudo-sample u (cached; malignancy-agnostic) ----
if M2_CLUSTERS_NPY.exists():
    clusters = np.load(M2_CLUSTERS_NPY, allow_pickle=True).astype(str)
    assert clusters.shape[0] == adata.n_obs
    adata.obs["cluster_m2"] = clusters
    print("loaded cached leiden clusters:", M2_CLUSTERS_NPY.name)
else:
    adata.obsm["X_mrvi_psm2_u"] = u
    sc.pp.neighbors(adata, use_rep="X_mrvi_psm2_u", random_state=SEED)   # HEAVY (~400k cells)
    sc.tl.leiden(adata, random_state=SEED, key_added="cluster_m2")
    clusters = adata.obs["cluster_m2"].astype(str).to_numpy()
    np.save(M2_CLUSTERS_NPY, clusters)
    print("computed + saved leiden clusters ->", M2_CLUSTERS_NPY.name)
print("clusters:", adata.obs["cluster_m2"].nunique())

mal_mask = (adata.obs["tcr_label"] == "malignant").to_numpy()
ben_mask = (adata.obs["tcr_label"] == "benign").to_numpy()
amask = mal_mask | ben_mask
ya = mal_mask[amask].astype(int)


def cluster_centroid_score(R, clusters, mal_mask, ben_mask):
    """s = ||R - ben_centroid[c]|| - ||R - mal_centroid[c]||, centroids per cluster from anchors;
    fall back to the global class centroid where a cluster lacks that class."""
    g_mal = R[mal_mask].mean(0); g_ben = R[ben_mask].mean(0)
    s = np.empty(R.shape[0], np.float32)
    for c in np.unique(clusters):
        cm = clusters == c
        m_in, b_in = cm & mal_mask, cm & ben_mask
        m_ref = R[m_in].mean(0) if m_in.any() else g_mal
        b_ref = R[b_in].mean(0) if b_in.any() else g_ben
        Rc = R[cm]
        s[cm] = np.linalg.norm(Rc - b_ref, axis=1) - np.linalg.norm(Rc - m_ref, axis=1)
    return s


# ---- score every cell + 1-D logistic on anchors -> prob_m2 ----
s = cluster_centroid_score(R, clusters, mal_mask, ben_mask)
lr = LogisticRegression().fit(s[amask].reshape(-1, 1), ya)
prob_m2 = lr.predict_proba(s.reshape(-1, 1))[:, 1]
print(f"score s range: [{s.min():.3f}, {s.max():.3f}]  |  anchors: {int(amask.sum())}")

## Validation — anchor CV (sets `thr_m2`)

`StratifiedGroupKFold` (5), groups = donor then study. Per fold the per-cluster centroids are re-derived from **train-donor anchors only**, the score is recomputed, and the 1-D logistic is refit on train anchors; held-out anchors are evaluated. (The MrVI model itself is trained once on all cells — fold re-centroiding removes centroid leakage, not model-training leakage; same limitation as M1.) Then the HC clamp, abstain flag, and merge into `calls.csv`.

In [ ]:
# anchor CV -> thr_m2 + final calls  ·  HEAVY-ish (10 centroid rescore passes) unless cached
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score

if (not REFIT_M2) and M2_CACHE.exists():
    zc = np.load(M2_CACHE, allow_pickle=True)
    assert list(zc["barcodes"]) == list(adata.obs_names.astype(str)), "M2 cache barcode mismatch -> set REFIT_M2=True"
    thr_m2 = float(zc["thr_m2"]); prob_m2 = zc["prob_m2"]; call_m2 = zc["call_m2"]; uncertain_m2 = zc["uncertain_m2"]
    prob_m2_oof = zc["prob_m2_oof"] if "prob_m2_oof" in zc.files else np.full(adata.n_obs, np.nan)
    print(f"loaded M2 cache -> thr_m2={thr_m2:.3f}  called malignant {int(call_m2.sum())}/{adata.n_obs}"
          f"  uncertain {int(uncertain_m2.sum())}  (set REFIT_M2=True to recompute)")
else:
    aidx = np.where(amask)[0]
    stage_a = adata.obs[stage_key].astype(str).to_numpy()[amask]
    study_a = adata.obs[STUDY_KEY].astype(str).to_numpy()[amask]

    def _best_f1_thr(y, p):
        best = (0.5, 0.0)
        for t in np.unique(np.quantile(p, np.linspace(0.05, 0.95, 19))):
            f = f1_score(y, (p >= t).astype(int), zero_division=0)
            if f > best[1]:
                best = (float(t), f)
        return best[0]

    def run_cv(group_key):
        """5-fold StratifiedGroupKFold; returns (median per-fold thr, pooled out-of-fold anchor probs)."""
        groups = adata.obs[group_key].astype(str).to_numpy()[amask]
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        oof = np.full(amask.sum(), np.nan); thrs = []
        for tr, te in sgkf.split(np.zeros(amask.sum()), ya, groups):
            tr_idx = aidx[tr]
            tr_mal = np.zeros(adata.n_obs, bool); tr_mal[tr_idx[ya[tr] == 1]] = True
            tr_ben = np.zeros(adata.n_obs, bool); tr_ben[tr_idx[ya[tr] == 0]] = True
            sm = cluster_centroid_score(R, clusters, tr_mal, tr_ben)   # centroids from train anchors only
            lrm = LogisticRegression().fit(sm[tr_idx].reshape(-1, 1), ya[tr])
            p = lrm.predict_proba(sm.reshape(-1, 1))[:, 1]
            thrs.append(_best_f1_thr(ya[tr], p[tr_idx]))
            oof[te] = p[aidx[te]]
        thr = float(np.median(thrs)); yhat = (oof >= thr).astype(int)
        def rep(mask, name):
            if mask.sum() < 20 or len(np.unique(ya[mask])) < 2:
                print(f"  {name:22s} n={int(mask.sum()):6d}  (too few / one class)"); return
            print(f"  {name:22s} n={int(mask.sum()):6d}  F1={f1_score(ya[mask], yhat[mask], zero_division=0):.3f}"
                  f"  AUC={roc_auc_score(ya[mask], oof[mask]):.3f}"
                  f"  balAcc={balanced_accuracy_score(ya[mask], yhat[mask]):.3f}")
        print(f"[CV grouped by {group_key}]  thr={thr:.3f}")
        rep(np.ones(len(ya), bool), "overall")
        for v in np.unique(stage_a): rep(stage_a == v, f"stage={v}")
        for v in np.unique(study_a): rep(study_a == v, f"study={v}")
        return thr, oof

    thr_m2, oof_m2 = run_cv(SAMPLE_KEY)        # donor-grouped -> sets the threshold + OOF validation preds
    _ = run_cv(STUDY_KEY)                      # study-grouped -> cross-study transfer report
    print("\nselected thr_m2 =", round(thr_m2, 3))

    # persist OOF (honest, held-out) anchor probs for the validation summary — NaN for non-anchors
    prob_m2_oof = np.full(adata.n_obs, np.nan); prob_m2_oof[aidx] = oof_m2

    # final calls + HC clamp + abstain
    call_m2 = (prob_m2 >= thr_m2).astype(int)
    hc = adata.obs[stage_key].astype(str).to_numpy() == "HC"
    call_m2[hc] = 0                                                       # HC clamp: known negatives -> benign
    uncertain_m2 = ((prob_m2 >= 0.4) & (prob_m2 <= 0.6)) | (np.abs(s) < MARGIN_MIN)
    np.savez(M2_CACHE, barcodes=np.array(adata.obs_names.astype(str)), thr_m2=thr_m2,
             prob_m2=prob_m2, call_m2=call_m2, uncertain_m2=uncertain_m2, prob_m2_oof=prob_m2_oof)
    print(f"called malignant: {int(call_m2.sum())}/{adata.n_obs}  |  HC clamped: {int(hc.sum())}"
          f"  |  uncertain: {int(uncertain_m2.sum())}  | saved M2 cache -> {M2_CACHE.name}")

adata.obs["prob_m2"] = prob_m2
adata.obs["call_m2"] = call_m2
adata.obs["uncertain_m2"] = uncertain_m2
adata.obs["prob_m2_oof"] = prob_m2_oof

# merge into shared calls.csv
calls = pd.read_csv(CALLS_CSV, index_col=0)
calls.loc[adata.obs_names, "prob_m2"] = prob_m2
calls.loc[adata.obs_names, "call_m2"] = call_m2
calls.loc[adata.obs_names, "prob_m2_oof"] = prob_m2_oof
calls.to_csv(CALLS_CSV)
print("merged prob_m2/call_m2/prob_m2_oof ->", CALLS_CSV)

In [ ]:
# nb17 UMAP panel on the u-derived UMAP: tcr_label | prob_m2 | call_m2
import matplotlib.pyplot as plt
UM = adata.obsm["X_umap_u"]; lab = adata.obs["tcr_label"].astype(str).values
TCRC = {"malignant": "#d62728", "benign": "#1f77b4", "unlabeled": "#dddddd"}
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for k in ["unlabeled", "benign", "malignant"]:
    m = lab == k
    ax[0].scatter(UM[m, 0], UM[m, 1], s=2, c=TCRC[k], label=k, linewidths=0)
ax[0].legend(markerscale=3, fontsize=7); ax[0].set_title("tcr_label")
s2 = ax[1].scatter(UM[:, 0], UM[:, 1], s=2, c=prob_m2, cmap="viridis", linewidths=0)
plt.colorbar(s2, ax=ax[1], fraction=0.04); ax[1].set_title("prob_m2")
cm = np.where(call_m2 == 1, "#d62728", "#1f77b4")
ax[2].scatter(UM[:, 0], UM[:, 1], s=2, c=cm, linewidths=0)
ax[2].set_title(f"call_m2 (thr={thr_m2:.2f})")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(OUT / "nb17_umap_panel.png", dpi=150); plt.show()


In [ ]:
# ---- cache the final M2 call for nb18 (bool methods parquet, indexed by obs_name) ----
MRVI_M2_PARQUET = NB_DIR / "data/atlas_joint/skin_T_mrvi_m2_malignancy_methods.parquet"
mcalls = (adata.obs[["call_m2"]]
          .rename(columns={"call_m2": "mrvi_m2_pseudosample"}).astype(bool))
mcalls.index.name = "obs_name"
mcalls.to_parquet(MRVI_M2_PARQUET)
print("wrote", MRVI_M2_PARQUET, mcalls.shape, int(mcalls["mrvi_m2_pseudosample"].sum()))

## Shared benchmarking (identical in nb16 / nb17)

Reloads `calls.csv` and renders whatever method columns exist so far: UMAP benchmark, per-cell track heatmap, concordance vs TCR anchors. Run nb16 then nb17 to populate both methods.

In [ ]:
# ============================ SHARED BENCHMARKING (identical in nb16/nb17) ============================
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score

calls = pd.read_csv(CALLS_CSV, index_col=0).reindex(adata.obs_names)
UM = adata.obsm["X_umap_u"]
TCRC = {"malignant": "#d62728", "benign": "#1f77b4", "unlabeled": "#dddddd"}
callmap = {0: "benign", 1: "malignant"}
has_m1 = ("call_m1" in calls.columns) and calls["call_m1"].notna().any()
has_m2 = ("call_m2" in calls.columns) and calls["call_m2"].notna().any()

# ---- A) UMAP benchmark ----
def _disc_scatter(ax, series, title, colors):
    order = [k for k in colors if k in ("unlabeled", "agree-benign")] + \
            [k for k in colors if k not in ("unlabeled", "agree-benign")]
    for k in order:
        m = (series == k).values
        if m.any():
            ax.scatter(UM[m, 0], UM[m, 1], s=2, c=colors[k], label=k, linewidths=0)
    ax.legend(markerscale=3, fontsize=6, loc="best"); ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

panels = [("tcr_label", calls["tcr_label"], TCRC)]
if has_m1: panels.append(("call_m1", calls["call_m1"].map(callmap), TCRC))
if has_m2: panels.append(("call_m2", calls["call_m2"].map(callmap), TCRC))
if has_m1 and has_m2:
    agree = np.where((calls.call_m1 == 1) & (calls.call_m2 == 1), "agree-malignant",
             np.where((calls.call_m1 == 0) & (calls.call_m2 == 0), "agree-benign", "disagree"))
    panels.append(("agreement", pd.Series(agree, index=calls.index),
                   {"agree-malignant": "#d62728", "agree-benign": "#1f77b4", "disagree": "#ff7f0e"}))
fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
axes = np.atleast_1d(axes)
for ax, (t, c, cols) in zip(axes, panels):
    _disc_scatter(ax, c, t, cols)
plt.tight_layout(); plt.savefig(OUT / "umap_benchmark.png", dpi=150); plt.show()

# ---- B) per-cell track heatmap ----
prob1 = calls["prob_m1"].values if "prob_m1" in calls.columns else np.full(len(calls), np.nan)
prob2 = calls["prob_m2"].values if "prob_m2" in calls.columns else np.full(len(calls), np.nan)
tcr_num = calls["tcr_label"].map({"malignant": 1.0, "benign": 0.0}).values   # unlabeled -> nan
clone_log = np.log1p(calls["clone_size"].fillna(0).values)
probs = [p for p in (prob1, prob2) if np.isfinite(p).any()]
ens = np.nanmean(np.vstack(probs), axis=0) if probs else clone_log

is_anchor = calls["tcr_label"].ne("unlabeled").values
keep = np.ones(len(calls), bool)
unl = np.where(~is_anchor)[0]
if len(unl) > 20000:
    drop = np.random.default_rng(SEED).choice(unl, len(unl) - 20000, replace=False)
    keep[drop] = False
idx = np.where(keep)[0]
order = idx[np.argsort(-ens[idx])]

tracks = [("prob_m1", prob1, "viridis"), ("prob_m2", prob2, "viridis"),
          ("tcr_status", tcr_num, "coolwarm"), ("clone_size (log1p)", clone_log, "magma")]
fig, axes = plt.subplots(len(tracks), 1, figsize=(14, 4), sharex=True)
for ax, (name, arr, cm) in zip(axes, tracks):
    im = ax.imshow(arr[order][None, :], aspect="auto", cmap=cm)
    ax.set_yticks([]); ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, pad=0.01, fraction=0.02)
axes[-1].set_xlabel("cells (sorted by ensemble malignant score)")
plt.tight_layout(); plt.savefig(OUT / "cell_track_heatmap.png", dpi=150); plt.show()

# ---- C) concordance vs TCR anchors ----
am = calls["tcr_label"].isin(["malignant", "benign"]).values
yt = (calls["tcr_label"] == "malignant").astype(int).values
rows = []
for name in ["call_m1", "call_m2"]:
    if name not in calls.columns or calls[name].isna().all():
        continue
    yp = calls[name].values.astype(float)
    m = am & ~np.isnan(yp)
    rows.append(dict(method=name, n_anchor=int(m.sum()),
                     precision=round(precision_score(yt[m], yp[m], zero_division=0), 3),
                     recall=round(recall_score(yt[m], yp[m], zero_division=0), 3),
                     f1=round(f1_score(yt[m], yp[m], zero_division=0), 3),
                     bal_acc=round(balanced_accuracy_score(yt[m], yp[m]), 3)))
conc = pd.DataFrame(rows)
print("concordance vs TCR anchors:\n", conc.to_string(index=False))
if has_m1 and has_m2:
    agreed = calls["call_m1"] == calls["call_m2"]
    print(f"\nM1<->M2 agreement overall: {agreed.mean():.3f}")
    for k in ["stage", STUDY_KEY]:
        print(f"\nM1<->M2 agreement by {k}:")
        print(calls.groupby(k).apply(lambda d: (d.call_m1 == d.call_m2).mean()).round(3).to_string())
conc.to_csv(OUT / "concordance_summary.csv", index=False)
print("\nwrote", OUT / "concordance_summary.csv")


## Validation trial — out-of-fold (honest) performance, both methods

The leakage-free read of each classifier vs the TCR anchors: per-fold held-out probabilities
(`prob_m1_oof` from nb16, `prob_m2_oof` from nb17), thresholded at each method's own OOF F1-optimal
cut. Confusion + precision / recall / F1 / balanced-accuracy / AUROC, overall and broken down by
stage and study. Run nb16 first so `prob_m1_oof` is in `calls.csv`.

In [ ]:
# ============================ VALIDATION — OOF (honest) for both methods ============================
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             balanced_accuracy_score, roc_auc_score, confusion_matrix)

cv = pd.read_csv(CALLS_CSV, index_col=0)
yt_all = (cv["tcr_label"] == "malignant").astype(int).to_numpy()
anchor = cv["tcr_label"].isin(["malignant", "benign"]).to_numpy()


def _best_f1_thr(y, p):
    best = (0.5, -1.0)
    for t in np.unique(np.quantile(p, np.linspace(0.05, 0.95, 19))):
        f = f1_score(y, (p >= t).astype(int), zero_division=0)
        if f > best[1]:
            best = (float(t), f)
    return best[0]


def _metrics(y, p, thr):
    yh = (p >= thr).astype(int)
    return dict(n=int(len(y)), thr=round(thr, 3),
                acc=round(accuracy_score(y, yh), 3),
                precision=round(precision_score(y, yh, zero_division=0), 3),
                recall=round(recall_score(y, yh, zero_division=0), 3),
                f1=round(f1_score(y, yh, zero_division=0), 3),
                balAcc=round(balanced_accuracy_score(y, yh), 3),
                AUC=round(roc_auc_score(y, p), 3) if len(np.unique(y)) > 1 else np.nan)


summary = []
for m, col in [("M1_labelspreading", "prob_m1_oof"), ("M2_pseudosample_mrvi", "prob_m2_oof")]:
    if col not in cv.columns or cv[col].notna().sum() == 0:
        print(f"{m}: {col} absent in calls.csv — run that notebook first"); continue
    p = cv[col].to_numpy()
    val = anchor & np.isfinite(p)
    y, pv = yt_all[val], p[val]
    thr = _best_f1_thr(y, pv)
    cm = confusion_matrix(y, (pv >= thr).astype(int), labels=[1, 0])
    print(f"\n=== {m} — OOF anchors (n={int(val.sum())}, thr={thr:.3f}) ===")
    print(pd.DataFrame(cm, index=["TCR malignant", "TCR benign"],
                       columns=["pred malignant", "pred benign"]).to_string())
    row = _metrics(y, pv, thr)
    summary.append({"scope": "overall", "method": m, **row})
    print("  overall:", {k: row[k] for k in ["acc", "precision", "recall", "f1", "balAcc", "AUC"]})
    for key in ["stage", STUDY_KEY]:
        g = cv[key].astype(str).to_numpy()[val]
        for v in sorted(np.unique(g)):
            mm = g == v
            if mm.sum() < 20 or len(np.unique(y[mm])) < 2:
                continue
            r = _metrics(y[mm], pv[mm], thr)
            summary.append({"scope": f"{key}={v}", "method": m, **r})
            print(f"    {key}={v:18s} acc={r['acc']:.3f}  f1={r['f1']:.3f}  AUC={r['AUC']:.3f}  balAcc={r['balAcc']:.3f}  (n={r['n']})")

val_df = pd.DataFrame(summary)
val_df.to_csv(OUT / "oof_validation_both_methods.csv", index=False)
print("\nwrote", OUT / "oof_validation_both_methods.csv")
print("\noverall quality table (both methods):")
print(val_df[val_df["scope"] == "overall"].to_string(index=False))
val_df[val_df["scope"] == "overall"]